# AprilTag Perception - Integration Test

Subscribe to AprilTag detection topic and display detected tags.

Prerequisites:
- Build: `colcon build --packages-select apriltag_interfaces apriltag_perception`
- Source: `source install/setup.bash`
- Launch camera: `export CYCLONEDDS_URI='file:///tmp/cyclonedds_eth1.xml' && ros2 launch camera_test_publisher camera_test_publisher.launch.py`
- Launch apriltag: `export CYCLONEDDS_URI='file:///tmp/cyclonedds_eth1.xml' && ros2 launch apriltag_perception apriltag_perception_test.launch.py`

In [ ]:
import ctypes
import os
from pathlib import Path
import sys

os.environ['CYCLONEDDS_URI'] = 'file:///tmp/cyclonedds_eth1.xml'

CWD = Path.cwd().resolve()
candidates = [CWD, CWD.parent, CWD.parent.parent]
PKG_ROOT = None
for c in candidates:
    if (c / 'apriltag_perception').exists() and (c / 'package.xml').exists():
        PKG_ROOT = c
        break
if PKG_ROOT is None:
    raise RuntimeError('Cannot locate apriltag_perception package root.')

WS_ROOT = PKG_ROOT.parents[2]
install_root = WS_ROOT / 'install'
_lib_dirs = []
if install_root.exists():
    for pkg_dir in install_root.iterdir():
        if not pkg_dir.is_dir():
            continue
        lib_dir = pkg_dir / 'lib'
        if lib_dir.exists():
            _lib_dirs.append(str(lib_dir.resolve()))
        for p in pkg_dir.glob('lib/python*/site-packages'):
            p_str = str(p.resolve())
            if p_str not in sys.path:
                sys.path.insert(0, p_str)

for ros_lib in ('/opt/ros/foxy/lib', '/opt/ros/foxy/lib/x86_64-linux-gnu'):
    _lib_dirs.append(ros_lib)
os.environ['LD_LIBRARY_PATH'] = ':'.join(_lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

_ai_lib_dir = install_root / 'apriltag_interfaces' / 'lib'
for _ai_so in ['libapriltag_interfaces__rosidl_generator_c.so',
               'libapriltag_interfaces__rosidl_typesupport_c.so',
               'libapriltag_interfaces__python.so']:
    _ai_so_path = str(_ai_lib_dir / _ai_so)
    if os.path.isfile(_ai_so_path):
        ctypes.CDLL(_ai_so_path, mode=ctypes.RTLD_GLOBAL)

import threading
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from cv_bridge import CvBridge

import rclpy
from rclpy.node import Node
from rclpy.qos import QoSProfile, ReliabilityPolicy
from sensor_msgs.msg import CameraInfo, Image
from apriltag_interfaces.msg import AprilTagDetection, AprilTagDetections


In [ ]:
CAPTURE_COUNT = 5
IMAGE_TOPIC = '/camera/test'
CAMERA_INFO_TOPIC = '/camera/test/camera_info'
DETECTION_TOPIC = '/perception/apriltag_detections/test'

collected_frames = []
collected_camera_info = []
collected_detections = []
bridge = CvBridge()
qos = QoSProfile(depth=10, reliability=ReliabilityPolicy.RELIABLE)

In [ ]:
class AprilTagCollector(Node):
    def __init__(self):
        super().__init__('apriltag_collector')
        self.image_sub = self.create_subscription(
            Image, IMAGE_TOPIC, self._image_callback, qos
        )
        self.info_sub = self.create_subscription(
            CameraInfo, CAMERA_INFO_TOPIC, self._info_callback, qos
        )
        self.detection_sub = self.create_subscription(
            AprilTagDetections, DETECTION_TOPIC, self._detection_callback, qos
        )
        self._latest_info = None

    def _image_callback(self, msg: Image):
        if len(collected_frames) >= CAPTURE_COUNT:
            return
        frame = bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
        collected_frames.append(frame)
        if self._latest_info is not None:
            collected_camera_info.append(self._latest_info)
        else:
            collected_camera_info.append(None)

    def _info_callback(self, msg: CameraInfo):
        self._latest_info = msg
        for i, info in enumerate(collected_camera_info):
            if info is None:
                collected_camera_info[i] = msg

    def _detection_callback(self, msg: AprilTagDetections):
        if len(collected_detections) >= CAPTURE_COUNT:
            return
        collected_detections.append(msg)
        tag_ids = [d.id for d in msg.detections]
        self.get_logger().info(
            f'Detection {len(collected_detections)}/{CAPTURE_COUNT}: '
            f'{len(msg.detections)} tag(s) {tag_ids}'
        )


if not rclpy.ok():
    rclpy.init()

node = AprilTagCollector()

def _spin():
    while rclpy.ok():
        rclpy.spin_once(node, timeout_sec=0.1)

spin_thread = threading.Thread(target=_spin, daemon=True)
spin_thread.start()
print(f'Collecting {CAPTURE_COUNT} frames and AprilTag detections ...')

In [ ]:
timeout = 5
start = time.time()
while len(collected_detections) < CAPTURE_COUNT and (time.time() - start) < timeout:
    time.sleep(0.5)

if len(collected_detections) < CAPTURE_COUNT:
    print(f'Timeout: only captured {len(collected_detections)}/{CAPTURE_COUNT} detections')
    print('Make sure both camera_test_publisher and apriltag_perception are running')
else:
    print(f'Successfully captured {len(collected_detections)} detection messages')

## Display captured frames with AprilTag overlays

In [ ]:
if collected_frames:
    n = len(collected_frames)
    cols = min(n, 5)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    for idx, frame in enumerate(collected_frames):
        r, c = divmod(idx, cols)
        ax = axes[r][c] if rows > 1 else axes[c]
        display_frame = frame.copy()
        if idx < len(collected_detections):
            for det in collected_detections[idx].detections:
                cx = int((det.center_offset_x + 1.0) * frame.shape[1] / 2.0)
                cy = int((det.center_offset_y + 1.0) * frame.shape[0] / 2.0)
                cv2.circle(display_frame, (cx, cy), 10, (0, 255, 0), 2)
                cv2.putText(display_frame, f'id={det.id}', (cx + 15, cy),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        rgb = cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB)
        ax.imshow(rgb)
        tag_count = len(collected_detections[idx].detections) if idx < len(collected_detections) else 0
        ax.set_title(f'Frame {idx + 1} ({tag_count} tags)')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No frames captured.')

## AprilTag detection details

In [ ]:
if collected_detections:
    total_tags = sum(len(msg.detections) for msg in collected_detections)
    print(f'Received {len(collected_detections)} detection messages, {total_tags} total tag detections')
    print(f'Frame ID: {collected_detections[0].frame_id}')
    print()
    for i, msg in enumerate(collected_detections):
        if msg.detections:
            print(f'Frame {i + 1}:')
            for det in msg.detections:
                print(f'  Tag id={det.id}  offset=({det.center_offset_x:+.3f}, {det.center_offset_y:+.3f})  '
                      f'dist={det.distance:.1f}mm  rpy=({det.roll:.1f}, {det.yaw:.1f}, {det.pitch:.1f})  '
                      f'hamming={det.hamming}')
        else:
            print(f'Frame {i + 1}: no tags detected')
else:
    print('No detections received.')

## Camera intrinsics

In [ ]:
if collected_camera_info and collected_camera_info[0] is not None:
    info = collected_camera_info[0]
    K = np.array(info.k).reshape(3, 3)
    D = np.array(info.d)
    P = np.array(info.p).reshape(3, 4)
    R = np.array(info.r).reshape(3, 3)

    print(f'Frame ID:   {info.header.frame_id}')
    print(f'Resolution: {info.width} x {info.height}')
    print(f'Distortion: {info.distortion_model}')
    print(f'D coeffs:   {D}')
    print(f'\nIntrinsic matrix K (3x3):')
    print(np.array2string(K, precision=2, suppress_small=True))
    print(f'\nProjection matrix P (3x4):')
    print(np.array2string(P, precision=2, suppress_small=True))
    print(f'\nRectification matrix R (3x3):')
    print(np.array2string(R, precision=2, suppress_small=True))
else:
    print('No camera info received.')

## Cleanup

In [ ]:
try:
    node.destroy_node()
except NameError:
    pass
if rclpy.ok():
    rclpy.shutdown()
print('Cleanup done.')